# Fetch DBU Prices from System Tables

This notebook queries `system.billing.list_prices` from **3 different workspaces** (AWS, Azure, GCP) and combines the results into a single table.

**Why 3 workspaces?**
- The `system.billing.list_prices` table is cloud-specific
- AWS workspace has AWS prices, Azure has Azure prices, etc.
- We need to query each workspace separately and combine

**Prerequisites:**
- Run `00_Setup_Secrets.ipynb` first to store workspace tokens


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_DBU_PRICES = f"{CATALOG}.{SCHEMA}.dbu_prices"
SECRET_SCOPE = "lakemeter-credentials"

print(f"✅ Target table: {TABLE_DBU_PRICES}")


In [ ]:
%pip install databricks-sdk -q


In [ ]:
import json
import pandas as pd
import requests
import time

# Load workspace configurations from secrets
def get_workspace_config(cloud):
    """Load workspace config from secret scope."""
    config_json = dbutils.secrets.get(SECRET_SCOPE, f"workspace-{cloud}")
    return json.loads(config_json)

# SQL query to fetch pricing
PRICING_QUERY = """
SELECT 
    sku_name,
    cloud,
    currency_code,
    usage_unit,
    pricing.default as price_per_dbu,
    price_start_time,
    price_end_time
FROM system.billing.list_prices
WHERE price_end_time IS NULL
ORDER BY sku_name
"""

def execute_sql_via_api(host, token, warehouse_id, sql, timeout_seconds=120):
    """Execute SQL via REST API (works across workspaces)."""
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }
    url = f"{host}/api/2.0/sql/statements"
    payload = {
        "warehouse_id": warehouse_id,
        "statement": sql,
        "wait_timeout": "50s",
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    result = response.json()
    
    statement_id = result.get("statement_id")
    status = result.get("status", {}).get("state")
    
    start_time = time.time()
    while status in ["PENDING", "RUNNING"]:
        if time.time() - start_time > timeout_seconds:
            raise TimeoutError(f"Query timed out after {timeout_seconds} seconds")
        time.sleep(2)
        poll_url = f"{host}/api/2.0/sql/statements/{statement_id}"
        poll_response = requests.get(poll_url, headers=headers)
        poll_response.raise_for_status()
        result = poll_response.json()
        status = result.get("status", {}).get("state")
    
    if status == "FAILED":
        error = result.get("status", {}).get("error", {})
        raise Exception(f"Query failed: {error}")
    
    return result

def api_response_to_dataframe(result):
    """Convert REST API response to pandas DataFrame."""
    manifest = result.get("manifest", {})
    data = result.get("result", {}).get("data_array", [])
    
    if not manifest or not data:
        return pd.DataFrame()
    
    columns = [col["name"] for col in manifest.get("schema", {}).get("columns", [])]
    return pd.DataFrame(data, columns=columns)

print("✅ Helper functions defined (REST API)")


In [ ]:
# Fetch pricing from all 3 workspaces via REST API
all_prices = []

for cloud in ["aws", "azure", "gcp"]:
    print(f"\n{'='*60}")
    print(f"📊 Fetching {cloud.upper()} prices via API...")
    print(f"{'='*60}")
    
    try:
        # Get config from secrets
        config = get_workspace_config(cloud)
        print(f"   Host: {config['host']}")
        print(f"   Warehouse: {config['warehouse_id']}")
        
        # Execute query via REST API
        print(f"   Executing query...")
        result = execute_sql_via_api(
            host=config["host"],
            token=config["token"],
            warehouse_id=config["warehouse_id"],
            sql=PRICING_QUERY
        )
        
        # Convert to DataFrame
        df = api_response_to_dataframe(result)
        
        if len(df) > 0:
            print(f"   ✅ Fetched {len(df)} pricing records")
            all_prices.append(df)
        else:
            print(f"   ⚠️ No pricing records found")
            
    except Exception as e:
        print(f"   ❌ Error: {e}")

print(f"\n{'='*60}")
print(f"✅ Total: {sum(len(df) for df in all_prices)} records from {len(all_prices)} workspaces")
print(f"{'='*60}")


In [ ]:
# Combine all pricing data
if all_prices:
    df_combined = pd.concat(all_prices, ignore_index=True)
    
    # NORMALIZE: Uppercase cloud column for consistency with other tables
    # system.billing.list_prices returns 'Azure' but our tables use 'AZURE'
    df_combined['cloud'] = df_combined['cloud'].str.upper()
    
    print(f"📊 Combined pricing data: {len(df_combined)} rows")
    print(f"\n📊 By Cloud (normalized to uppercase):")
    print(df_combined.groupby('cloud').size())
    
    print(f"\n📊 Sample data (first 20 rows):")
    display(df_combined.head(20))
else:
    print("❌ No pricing data fetched")


In [ ]:
# Show all unique SKU names
print(f"📋 Unique SKU names ({len(df_combined['sku_name'].unique())}):")
for sku in sorted(df_combined['sku_name'].unique()):
    print(f"   {sku}")


In [ ]:
# Show full data by cloud (using uppercase to match actual values)
print("📊 AWS Prices:")
display(df_combined[df_combined['cloud'] == 'AWS'])

print("\n📊 Azure Prices:")
display(df_combined[df_combined['cloud'] == 'AZURE'])

print("\n📊 GCP Prices:")
display(df_combined[df_combined['cloud'] == 'GCP'])


In [ ]:
# =============================================================================
# PARSE SKU NAMES - Extract tier, product_type, region
# =============================================================================

import re

# Known regions (order matters - longest first for matching)
KNOWN_REGIONS = [
    # AWS regions
    "US_EAST_N_VIRGINIA", "US_EAST_OHIO", "US_WEST_OREGON", "US_WEST_CALIFORNIA",
    "US_SOUTH_CAROLINA", "US_IOWA", "US_NEVADA", "US_VIRGINIA", "US_OREGON",
    "AP_MUMBAI", "AP_SINGAPORE", "AP_SYDNEY", "AP_TOKYO", "AP_SEOUL", "AP_JAKARTA",
    "EUROPE_IRELAND", "EUROPE_FRANKFURT", "EUROPE_LONDON", "EUROPE_FRANCE", 
    "EUROPE_BELGIUM", "EUROPE_ENGLAND", "EUROPE_STOCKHOLM",
    "CANADA", "CANADA_QUEBEC", "SA_BRAZIL", "ME_DAMMAM", "INDIA_MUMBAI",
    
    # Azure regions
    "US_EAST", "US_EAST_2", "US_WEST", "US_WEST_2", "US_WEST_3", 
    "US_CENTRAL", "US_NORTH_CENTRAL", "US_SOUTH_CENTRAL", "US_WEST_CENTRAL",
    "EU_WEST", "EU_NORTH", "UK_SOUTH", "UK_WEST",
    "FRANCE_CENTRAL", "GERMANY_WEST_CENTRAL", "SWITZERLAND_NORTH", "SWITZERLAND_WEST",
    "SWEDEN_CENTRAL", "NORWAY_EAST", "QATAR_CENTRAL", "UAE_NORTH",
    "ASIA_EAST", "ASIA_SOUTHEAST", "ASIA_SINGAPORE", "ASIA_TOKYO",
    "AUSTRALIA_EAST", "AUSTRALIA_SOUTHEAST", "AUSTRALIA_CENTRAL", "AUSTRALIA_CENTRAL_2",
    "AUSTRALIA_SYDNEY",
    "JAPAN_EAST", "JAPAN_WEST", "KOREA_CENTRAL",
    "INDIA_CENTRAL", "INDIA_SOUTH", "INDIA_WEST",
    "BRAZIL_SOUTH", "SOUTH_AFRICA_NORTH", "MEXICO_CENTRAL",
    "CANADA_CENTRAL", "CANADA_EAST",
    
    # GCP regions  
    "ASIA_SINGAPORE", "ASIA_TOKYO", "AUSTRALIA_SYDNEY", 
    "EUROPE_BELGIUM", "EUROPE_ENGLAND",
    "US_IOWA", "US_NEVADA", "US_SOUTH_CAROLINA", "US_VIRGINIA",
]

# Tiers
TIERS = ["ENTERPRISE", "PREMIUM", "STANDARD", "MCT"]

def parse_sku_name(sku_name):
    """Parse SKU name into tier, product_type, and region."""
    
    # Extract tier
    tier = None
    remaining = sku_name
    for t in TIERS:
        if sku_name.startswith(t + "_"):
            tier = t
            remaining = sku_name[len(t) + 1:]
            break
    
    # Extract region (check from end)
    region = None
    for r in sorted(KNOWN_REGIONS, key=len, reverse=True):  # Longest first
        if remaining.endswith("_" + r):
            region = r
            remaining = remaining[:-(len(r) + 1)]
            break
    
    # Remaining is the product type
    product_type = remaining
    
    return tier, product_type, region

# Apply parsing to the combined dataframe
df_parsed = df_combined.copy()
df_parsed[['tier', 'product_type', 'region']] = df_parsed['sku_name'].apply(
    lambda x: pd.Series(parse_sku_name(x))
)

print(f"✅ Parsed {len(df_parsed)} SKU records")
print(f"\n📊 By Tier:")
print(df_parsed['tier'].value_counts())

print(f"\n📊 Unique Product Types ({df_parsed['product_type'].nunique()}):")
for pt in sorted(df_parsed['product_type'].unique()):
    print(f"   {pt}")

print(f"\n📊 Unique Regions ({df_parsed['region'].nunique()}):")


In [ ]:
# Display parsed data
print("📊 Parsed SKU Data (sample):")
display(df_parsed[['sku_name', 'cloud', 'tier', 'product_type', 'region', 'price_per_dbu', 'usage_unit']].head(30))

# Show global SKUs (no region)
print("\n📊 Global SKUs (no region):")
global_skus = df_parsed[df_parsed['region'].isna()][['sku_name', 'cloud', 'tier', 'product_type', 'price_per_dbu']].drop_duplicates()
display(global_skus)


In [ ]:
# =============================================================================
# SUMMARY: All SKUs by usage_unit and product_type
# =============================================================================

# Convert price to float
df_parsed['price_per_dbu'] = df_parsed['price_per_dbu'].astype(float)

print(f"📊 Total SKUs: {len(df_parsed)}")

print(f"\n📊 By Usage Unit:")
print(df_parsed['usage_unit'].value_counts())

print(f"\n📊 By Cloud:")
print(df_parsed['cloud'].value_counts())

print(f"\n📊 By Tier:")
print(df_parsed['tier'].value_counts())

print(f"\n📊 Unique Product Types: {df_parsed['product_type'].nunique()}")
print(f"📊 Unique Regions: {df_parsed['region'].nunique()} (+ global SKUs with no region)")


In [ ]:
# =============================================================================
# REGION MAPPING: SKU region names → databricks_regions.region (by cloud)
# =============================================================================

# Cloud-specific region mappings
AWS_REGION_MAPPING = {
    # US regions
    "US_EAST_N_VIRGINIA": "us-east-1",
    "US_EAST_OHIO": "us-east-2", 
    "US_WEST_OREGON": "us-west-2",
    "US_WEST_CALIFORNIA": "us-west-1",
    "US_OREGON": "us-west-2",
    "US_VIRGINIA": "us-east-1",
    
    # Asia Pacific
    "AP_MUMBAI": "ap-south-1",
    "AP_SINGAPORE": "ap-southeast-1",
    "AP_SYDNEY": "ap-southeast-2",
    "AP_TOKYO": "ap-northeast-1",
    "AP_SEOUL": "ap-northeast-2",
    "AP_JAKARTA": "ap-southeast-3",
    "ASIA_SINGAPORE": "ap-southeast-1",
    "ASIA_TOKYO": "ap-northeast-1",
    
    # Europe
    "EUROPE_IRELAND": "eu-west-1",
    "EUROPE_FRANKFURT": "eu-central-1",
    "EUROPE_LONDON": "eu-west-2",
    "EUROPE_FRANCE": "eu-west-3",
    "EUROPE_STOCKHOLM": "eu-north-1",
    
    # Canada
    "CANADA": "ca-central-1",
    "CANADA_QUEBEC": "ca-central-1",
    
    # South America
    "SA_BRAZIL": "sa-east-1",
    
    # Middle East
    "ME_DAMMAM": "me-central-1",
    
    # India (alias)
    "INDIA_MUMBAI": "ap-south-1",
    
    # Australia (alias)
    "AUSTRALIA_SYDNEY": "ap-southeast-2",
}

AZURE_REGION_MAPPING = {
    "US_EAST": "eastus",
    "US_EAST_2": "eastus2",
    "US_WEST": "westus",
    "US_WEST_2": "westus2",
    "US_WEST_3": "westus3",
    "US_CENTRAL": "centralus",
    "US_NORTH_CENTRAL": "northcentralus",
    "US_SOUTH_CENTRAL": "southcentralus",
    "US_WEST_CENTRAL": "westcentralus",
    "EU_WEST": "westeurope",
    "EU_NORTH": "northeurope",
    "UK_SOUTH": "uksouth",
    "UK_WEST": "ukwest",
    "FRANCE_CENTRAL": "francecentral",
    "GERMANY_WEST_CENTRAL": "germanywestcentral",
    "SWITZERLAND_NORTH": "switzerlandnorth",
    "SWITZERLAND_WEST": "switzerlandwest",
    "SWEDEN_CENTRAL": "swedencentral",
    "NORWAY_EAST": "norwayeast",
    "QATAR_CENTRAL": "qatarcentral",
    "UAE_NORTH": "uaenorth",
    "ASIA_EAST": "eastasia",
    "ASIA_SOUTHEAST": "southeastasia",
    "ASIA_SINGAPORE": "southeastasia",
    "ASIA_TOKYO": "japaneast",
    "AUSTRALIA_EAST": "australiaeast",
    "AUSTRALIA_SOUTHEAST": "australiasoutheast",
    "AUSTRALIA_CENTRAL": "australiacentral",
    "AUSTRALIA_CENTRAL_2": "australiacentral2",
    "AUSTRALIA_SYDNEY": "australiaeast",
    "JAPAN_EAST": "japaneast",
    "JAPAN_WEST": "japanwest",
    "KOREA_CENTRAL": "koreacentral",
    "INDIA_CENTRAL": "centralindia",
    "INDIA_SOUTH": "southindia",
    "INDIA_WEST": "westindia",
    "BRAZIL_SOUTH": "brazilsouth",
    "SOUTH_AFRICA_NORTH": "southafricanorth",
    "MEXICO_CENTRAL": "mexicocentral",
    "CANADA_CENTRAL": "canadacentral",
    "CANADA_EAST": "canadaeast",
}

GCP_REGION_MAPPING = {
    # US regions
    "US_IOWA": "us-central1",
    "US_NEVADA": "us-west4",             # Las Vegas, Nevada
    "US_SOUTH_CAROLINA": "us-east1",
    "US_OREGON": "us-west1",             # The Dalles, Oregon
    "US_VIRGINIA": "us-east4",
    "US_WEST_OREGON": "us-west1",
    "US_WEST_CALIFORNIA": "us-west2",    # Los Angeles, California
    "US_EAST_N_VIRGINIA": "us-east4",
    
    # Europe regions
    "EUROPE_BELGIUM": "europe-west1",
    "EUROPE_ENGLAND": "europe-west2",
    "EUROPE_FRANKFURT": "europe-west3",
    "EUROPE_FRANCE": "europe-west9",
    "EUROPE_LONDON": "europe-west2",
    "EUROPE_IRELAND": "europe-west1",
    
    # Canada
    "CANADA_QUEBEC": "northamerica-northeast1",
    "CANADA": "northamerica-northeast1",
    
    # Asia Pacific
    "ASIA_SINGAPORE": "asia-southeast1",
    "ASIA_TOKYO": "asia-northeast1",
    "AP_SINGAPORE": "asia-southeast1",
    "AP_TOKYO": "asia-northeast1",
    "AP_SYDNEY": "australia-southeast1",
    "AP_MUMBAI": "asia-south1",
    "AP_SEOUL": "asia-northeast3",
    
    # Australia
    "AUSTRALIA_SYDNEY": "australia-southeast1",
    
    # India
    "INDIA_MUMBAI": "asia-south1",
    
    # South America
    "SA_BRAZIL": "southamerica-east1",
    
    # Middle East
    "ME_DAMMAM": "me-central2",  # Dammam, Saudi Arabia
}

# Combined mapping for backward compatibility (used in df_parsed mapping)
REGION_MAPPING = {}
REGION_MAPPING.update(AWS_REGION_MAPPING)
REGION_MAPPING.update(AZURE_REGION_MAPPING)
REGION_MAPPING.update(GCP_REGION_MAPPING)

# Create the mapping dataframe WITH cloud column
region_mapping_data = []
for sku_region, region_code in AWS_REGION_MAPPING.items():
    region_mapping_data.append({"cloud": "AWS", "sku_region": sku_region, "region_code": region_code})
for sku_region, region_code in AZURE_REGION_MAPPING.items():
    region_mapping_data.append({"cloud": "AZURE", "sku_region": sku_region, "region_code": region_code})
for sku_region, region_code in GCP_REGION_MAPPING.items():
    region_mapping_data.append({"cloud": "GCP", "sku_region": sku_region, "region_code": region_code})

df_region_mapping = pd.DataFrame(region_mapping_data)

print(f"📊 Region Mapping Table: {len(df_region_mapping)} mappings")
print(f"   AWS: {len(AWS_REGION_MAPPING)}, AZURE: {len(AZURE_REGION_MAPPING)}, GCP: {len(GCP_REGION_MAPPING)}")
display(df_region_mapping)


In [ ]:
# =============================================================================
# APPLY REGION MAPPING: Add region_code column
# =============================================================================

# Map SKU region to actual region code
df_parsed['region_code'] = df_parsed['region'].map(REGION_MAPPING)

# Check for unmapped regions
unmapped = df_parsed[df_parsed['region'].notna() & df_parsed['region_code'].isna()]['region'].unique()
if len(unmapped) > 0:
    print(f"⚠️ Unmapped regions ({len(unmapped)}):")
    for r in sorted(unmapped):
        print(f"   {r}")
else:
    print("✅ All regions mapped successfully!")

# Show mapping coverage
total_with_region = len(df_parsed[df_parsed['region'].notna()])
mapped = len(df_parsed[df_parsed['region_code'].notna()])
print(f"\n📊 Mapping Coverage:")
print(f"   SKUs with region: {total_with_region}")
print(f"   Successfully mapped: {mapped}")
print(f"   Coverage: {mapped/total_with_region*100:.1f}%")

# Show sample with region_code
print("\n📊 Sample with region_code:")
display(df_parsed[df_parsed['region'].notna()][['sku_name', 'cloud', 'tier', 'product_type', 'region', 'region_code', 'price_per_dbu']].head(20))


In [ ]:
# =============================================================================
# SAVE REGION MAPPING TABLE
# =============================================================================

TABLE_REGION_MAPPING = f"{CATALOG}.{SCHEMA}.sku_region_mapping"

# Save region mapping to table
spark_df_mapping = spark.createDataFrame(df_region_mapping)
spark_df_mapping.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_REGION_MAPPING)

print(f"✅ Saved region mapping to {TABLE_REGION_MAPPING}")

# Verify
display(spark.sql(f"SELECT * FROM {TABLE_REGION_MAPPING} ORDER BY sku_region"))


In [ ]:
# =============================================================================
# SAVE DBU PRICES TABLE
# =============================================================================

# Select and rename columns for the final table
df_final = df_parsed[[
    'sku_name', 'cloud', 'tier', 'product_type', 'region', 'region_code',
    'usage_unit', 'price_per_dbu', 'currency_code', 'price_start_time'
]].copy()

# Ensure price_per_dbu is float
df_final['price_per_dbu'] = df_final['price_per_dbu'].astype(float)

# Add fetched timestamp
from datetime import datetime
df_final['fetched_at'] = datetime.utcnow().isoformat()

# Save to table
spark_df_prices = spark.createDataFrame(df_final)
spark_df_prices.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_DBU_PRICES)

print(f"✅ Saved {len(df_final)} pricing records to {TABLE_DBU_PRICES}")

# Verify
print(f"\n📊 Sample from saved table:")
display(spark.sql(f"SELECT * FROM {TABLE_DBU_PRICES} LIMIT 10"))


In [ ]:
# =============================================================================
# VERIFY: Show all unique product_type values (for joining with serverless_product_rates)
# =============================================================================

print("📊 All unique product_type values in dbu_prices table:")
print("=" * 60)

product_types_df = spark.sql(f"""
    SELECT DISTINCT 
        product_type,
        COUNT(*) as sku_count,
        COLLECT_SET(cloud) as clouds,
        COLLECT_SET(usage_unit) as usage_units
    FROM {TABLE_DBU_PRICES}
    GROUP BY product_type
    ORDER BY product_type
""")

display(product_types_df)
